# 08: Visual Comparison of Pointcloud Labeling Methods

This notebook provides a modular and reproducible framework to visually compare different strategies for projecting 2D segmentation masks onto a 3D pointcloud, using Project Aria data.

### In this notebook, we will:
1. Data loading and setup
2. Mask extraction utilities (SAM2)
3. Baseline projection: single frame, single camera
4. Accumulated projection: multi-frame
5. Multi-camera projection
6. Visual comparison of results

As a sample, we will use `kettle_and_forklift_recording.vrs` located in the local `data/raw/kettle_and_forklift/` directory and its corresponding segmentation masks located in `data/outputs/segmentation/kettle_and_forklift/sam2/masks` .


## 1. Data Loading and Setup

In this section, we load the 3D pointcloud and camera trajectory data produced by MPS

In [1]:
# Import required libraries and load pointcloud and trajectory data
import os
import pandas as pd
from aria_pylib import find_points_file, find_trajectory_file

# Define the directory containing MPS outputs
MPS_DIR = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'mps_kettle_and_forklift_recording_vrs', 'slam')

# Load the pointcloud file
points_path = find_points_file(MPS_DIR)
print(f"Loading pointcloud from: {points_path}")
points_df = pd.read_csv(points_path)

# Load trajectory file
trajectory_path = find_trajectory_file(MPS_DIR)
print(f"Loading trajectory from: {trajectory_path}")
trajectory_df = pd.read_csv(trajectory_path)
print(f"Loaded {len(trajectory_df)} poses.")

Loading pointcloud from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\semidense_points.csv.gz
Loading trajectory from: ..\data\raw\kettle_and_forklift\mps_kettle_and_forklift_recording_vrs\slam\closed_loop_trajectory.csv
Loaded 50626 poses.


## 2. Mask Extraction Utilities

This section provides utility functions to extract 2D segmentation masks from the dataset, given a specific timestamp and camera. These functions will be used to retrieve the masks needed for each projection experiment.

In [ ]:
# Utility function to load a segmentation mask for a given frame and camera
import cv2

def load_mask(masks_dir, frame_idx, camera_name="camera-rgb"):
    """
    Loads a 2D segmentation mask for the specified frame and camera.
    """
    mask_filename = f"{frame_idx:06d}.png"
    mask_path = os.path.join(masks_dir, camera_name, mask_filename) if os.path.isdir(os.path.join(masks_dir, camera_name)) else os.path.join(masks_dir, mask_filename)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(f"Could not load mask at {mask_path}. Check if the file exists!")
    print(f"Loaded mask from {mask_path}\nShape: {mask.shape}")
    return mask

# example
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
frame_idx = 50
mask = load_mask(masks_dir, frame_idx)

Loaded mask from ..\data\outputs\segmentation\kettle_and_forklift\sam2\masks\000050.png
shape: (1408, 1408)


## 3. Baseline Projection: Single Frame, Single Camera

In this section, we implement the baseline projection method: projecting the 3D pointcloud onto a single 2D segmentation mask (from the RGB camera) at a specific frame. The resulting labeled pointcloud will be used as a reference for comparison with more advanced strategies.
We will use the frame 50 (video at 10 fps) as an example

In [ ]:
# Project 3D points onto a single 2D mask (RGB, single frame)
from projectaria_tools.core import data_provider
from scipy.spatial.transform import Rotation as R
import numpy as np

# --- Setup calibration and mask path ---
vrs_path = os.path.join('..', 'data', 'raw', 'kettle_and_forklift', 'kettle_and_forklift_recording.vrs')
masks_dir = os.path.join('..', 'data', 'outputs', 'segmentation', 'kettle_and_forklift', 'sam2', 'masks')
frame_idx = 50  # Example frame for visualization

# Load calibration from VRS and compute transform from device to camera
provider = data_provider.create_vrs_data_provider(vrs_path)
device_calib = provider.get_device_calibration()
rgb_calib = device_calib.get_camera_calib("camera-rgb")
T_Device_Camera = rgb_calib.get_transform_device_camera()

# Load the mask for the selected frame
mask = load_mask(masks_dir, frame_idx)

# --- Projection function ---
def project_world_to_rgb_image(point_world, pose_row, rgb_calib, T_Device_Camera):
    """
    Projects a 3D point from world coordinates to RGB pixel coordinates.
    Handles T_World_Device (MPS), T_Device_Camera (Extrinsics), and Distortion (Intrinsics).
    """
    quat = [pose_row['qx_world_device'], pose_row['qy_world_device'], pose_row['qz_world_device'], pose_row['qw_world_device']]
    rot_world_device = R.from_quat(quat).as_matrix()
    t_world_device = np.array([pose_row['tx_world_device'], pose_row['ty_world_device'], pose_row['tz_world_device']])
    point_device = rot_world_device.T @ (point_world - t_world_device)
    T_Camera_Device = T_Device_Camera.inverse()
    point_cam = T_Camera_Device @ point_device
    pixel = rgb_calib.project(point_cam)
    return pixel

# --- Label only points observed at the selected camera pose ---
def label_pointcloud_at_frame(points_df, trajectory_df, frame_idx, rgb_calib, T_Device_Camera, mask, segmentation_fps=10, trajectory_fps=30):
    """
    Label only the points that are visible from the camera pose corresponding to the selected frame.
    """
    segmentation_time = frame_idx / segmentation_fps
    trajectory_times = trajectory_df['timestamp_ns'] * 1e-9 if 'timestamp_ns' in trajectory_df.columns else np.arange(len(trajectory_df)) / trajectory_fps
    closest_traj_idx = (np.abs(trajectory_times - segmentation_time)).argmin()
    pose_row = trajectory_df.iloc[closest_traj_idx]
    labels = []
    px, py, pz = [], [], []
    for _, row in points_df.iterrows():
        point_3d = np.array([row['px_world'], row['py_world'], row['pz_world']])
        pixel = project_world_to_rgb_image(point_3d, pose_row, rgb_calib, T_Device_Camera)
        label = 0
        if pixel is not None:
            u, v = int(round(pixel[0])), int(round(pixel[1]))
            if 0 <= v < mask.shape[0] and 0 <= u < mask.shape[1]:
                label = mask[v, u]
        labels.append(label)
        px.append(point_3d[0])
        py.append(point_3d[1])
        pz.append(point_3d[2])
    labeled_df = pd.DataFrame({'px_world': px, 'py_world': py, 'pz_world': pz, 'semantic_label': labels})
    return labeled_df

# Run the baseline projection
labeled_points_frame = label_pointcloud_at_frame(points_df, trajectory_df, frame_idx, rgb_calib, T_Device_Camera, mask)
unique_labels_frame = np.unique(labeled_points_frame['semantic_label'])
print(f"Found unique labels in 3D for frame {frame_idx}: {unique_labels_frame}")